# <font color="steelblue">Segmentación de turistas</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

> **Guión de trabajo para el estudiante.** Este cuaderno es el **enunciado** de un proyecto de **aprendizaje no supervisado** que debéis **desarrollar vosotros**: objetivos, fases, tareas obligatorias, andamiaje mínimo de código  y criterios de evaluación. **No es una solución.**

In [ ]:
# @title Cargar configuración del cuaderno
!pip install gdown
!pip install --upgrade kagglehub
!pip install lightgbm xgboost
!pip install ucimlrepo

# Análisis numérico
import numpy as np
import pandas as pd
import math, random
import warnings
warnings.filterwarnings('ignore')

# Gráficos
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_theme(style='whitegrid')
%config InlineBackend.figure_format = 'retina'

import os, zipfile, gdown, kagglehub

# Funciones del curso
from urllib.request import urlretrieve
for fichero in ['preprocesar.py', 'evaluar_clasificadores.py', 'pca_funciones.py', 'auto_ML.py']:
    urlretrieve(f'https://raw.githubusercontent.com/ia4legos/MachineLearning/refs/heads/main/{fichero}', fichero)
from preprocesar import *
from evaluar_clasificadores import *
from pca_funciones import *
from auto_ML import *

from sklearn.decomposition import PCA, KernelPCA, IncrementalPCA, FastICA
from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans
from scipy.spatial.distance import cdist, pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, FeatureAgglomeration
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS, Isomap, LocallyLinearEmbedding, TSNE, SpectralEmbedding

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, calinski_harabasz_score, davies_bouldin_score,
                             accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             adjusted_rand_score)

RNG = 42

## Cargar funciones accesorias

############# Funciones MDS ##############
def stress1(D_orig, embedding):
    """Stress-1 de Kruskal entre las distancias originales y las del embedding.
    Portable entre versiones de scikit-learn (no depende del atributo stress_)."""
    D_proj = pairwise_distances(embedding)
    iu = np.triu_indices_from(D_orig, k=1)          # pares (i<j), sin diagonal
    num = np.sum((D_orig[iu] - D_proj[iu])**2)
    den = np.sum(D_proj[iu]**2)
    return np.sqrt(num / den)

def ajustar_mds(D, n_components=2, metrico=True, n_init=4, random_state=RNG):
    """Ajusta un MDS (métrico o no) sobre una matriz de distancias precalculada
    y devuelve el embedding. Usa 'metric'/'dissimilarity' por compatibilidad amplia."""
    modelo = MDS(n_components=n_components, dissimilarity='precomputed',
                 metric=metrico, n_init=n_init, random_state=random_state)
    return modelo.fit_transform(D)

def curva_stress(D, metrico=True, max_dim=10, n_init=4):
    """Calcula el Stress-1 para soluciones de 1 a max_dim dimensiones."""
    dims = range(1, max_dim + 1)
    valores = [stress1(D, ajustar_mds(D, d, metrico, n_init)) for d in dims]
    return list(dims), valores

def grafico_codo(dims, valores, titulo=''):
    """Curva de Stress-1 vs nº de dimensiones, con las bandas de calidad de Kruskal."""
    plt.figure(figsize=(8, 5))
    plt.plot(dims, valores, 'o-')
    for y, txt, col in [(0.05, 'bueno', 'green'), (0.10, 'aceptable', 'orange'), (0.20, 'pobre', 'red')]:
        plt.axhline(y, ls='--', lw=1, color=col, alpha=.6)
        plt.text(dims[-1], y, f'  {txt}', color=col, va='center')
    plt.xticks(dims); plt.xlabel('Nº de dimensiones'); plt.ylabel('Stress-1')
    plt.title(titulo); plt.show()

def grafico_distancias(D_orig, embedding, titulo='Distancias proyectadas vs originales'):
    """Gráfico distancia-distancia: nube sobre la diagonal => buen ajuste."""
    D_proj = pairwise_distances(embedding)
    m = np.max(D_orig)
    plt.figure(figsize=(8, 6))
    plt.scatter(D_orig, D_proj, s=8, alpha=.3, label='pares de puntos')
    plt.plot([0, m], [0, m], 'k-', lw=2, label='ajuste perfecto (y = x)')
    plt.xlabel('Distancia original'); plt.ylabel('Distancia proyectada')
    plt.title(titulo); plt.legend(); plt.show()

def nearest_neighbors(X, k):
    """Matriz (n, k) con los índices de los k vecinos más próximos de cada punto
    (distancia euclídea), excluyendo el propio punto. Devuelve enteros."""
    D = pairwise_distances(X)
    # ordena cada fila por distancia; la columna 0 es el propio punto (dist. 0)
    return np.argsort(D, axis=1)[:, 1:k+1]

def dibujar_grafo(X, knn, ax=None, **kw):
    """Dibuja las aristas punto–vecino sobre un eje 2D."""
    ax = ax or plt.gca()
    for i, vecinos in enumerate(knn):
        for j in vecinos:
            ax.plot(X[[i, j], 0], X[[i, j], 1], c='gray', lw=0.6, zorder=1, **kw)

########### Cluster ##############
def plot_dendrogram(model, **kwargs):
    """Dibuja el dendrograma a partir de un AgglomerativeClustering ya ajustado
    con compute_distances=True (o distance_threshold). Acepta los mismos kwargs
    que scipy.cluster.hierarchy.dendrogram (truncate_mode, p, color_threshold, ax...)."""
    counts = np.zeros(model.children_.shape[0])     # nº de muestras bajo cada fusión
    n = len(model.labels_)
    for i, merge in enumerate(model.children_):
        c = 0
        for child in merge:
            c += 1 if child < n else counts[child - n]   # hoja (1) o sub-cluster (su recuento)
        counts[i] = c
    Z = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    return dendrogram(Z, **kwargs)

def plot_scree(X, metric, linkage_m, kmin, kmax, **kwargs):
    """Índice de silueta para soluciones jerárquicas de kmin a kmax grupos."""
    ks = range(kmin, kmax)
    sil = [silhouette_score(X, AgglomerativeClustering(n_clusters=k, metric=metric,
                                                       linkage=linkage_m).fit_predict(X)) for k in ks]
    plt.plot(list(ks), sil, marker='o', **kwargs)
    plt.xlabel('número de grupos'); plt.ylabel('silueta media'); plt.xticks(list(ks))

from matplotlib.patches import Ellipse
def draw_ellipse(position, cov2x2, ax=None, **kwargs):
    """Dibuja la elipse (a 1, 2 y 3 sigmas) dada una covarianza 2x2."""
    ax = ax or plt.gca()
    U, s, Vt = np.linalg.svd(cov2x2)
    angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
    width, height = 2 * np.sqrt(s)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle=angle, **kwargs))   # 'angle' es keyword en matplotlib actual

def _cov_2x2(gmm, k):
    """Reconstruye la covarianza 2x2 de la componente k según covariance_type."""
    ct = gmm.covariance_type
    if ct == 'full':      return gmm.covariances_[k]
    if ct == 'tied':      return gmm.covariances_
    if ct == 'diag':      return np.diag(gmm.covariances_[k])
    return np.eye(2) * gmm.covariances_[k]            # 'spherical'

def plot_gmm(gmm, X, label=True, ax=None):
    """Ajusta el GMM y dibuja puntos coloreados por componente + sus elipses
    (válido para los cuatro covariance_type)."""
    ax = ax or plt.gca()
    labels = gmm.fit(X).predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels if label else None,
               s=20, cmap='viridis', zorder=2)
    ax.set_aspect('equal')
    w_factor = 0.3 / gmm.weights_.max()
    for k, (pos, w) in enumerate(zip(gmm.means_, gmm.weights_)):
        draw_ellipse(pos, _cov_2x2(gmm, k), ax=ax, alpha=w * w_factor, color='steelblue')

def draw_ellipsoid(ax, mean, cov, nsig=2, **kw):
    # dibuja el elipsoide a 'nsig' sigmas de una gaussiana 3D (media + cov 3x3)
    vals, vecs = np.linalg.eigh(cov)                 # autovalores/autovectores
    u = np.linspace(0, 2*np.pi, 24); v = np.linspace(0, np.pi, 16)
    esfera = np.array([np.outer(np.cos(u), np.sin(v)),
                       np.outer(np.sin(u), np.sin(v)),
                       np.outer(np.ones_like(u), np.cos(v))])
    r = nsig * np.sqrt(vals)                          # longitudes de los semiejes
    pts = np.einsum('ij,jkl->ikl', vecs, esfera * r[:, None, None])
    ax.plot_surface(pts[0]+mean[0], pts[1]+mean[1], pts[2]+mean[2], **kw)

## <font color="steelblue">Objetivos del proyecto</font>

A partir de los perfiles de valoración de usuarios de Google sobre 24 categorías de atracciones turísticas, **descubrir segmentos de turistas** con intereses parecidos y **caracterizarlos**. No hay variable respuesta: es un problema de **agrupación (clustering)** y **reducción de dimensión**. Al terminar, debéis ser capaces de:

* **Reducir la dimensión** e interpretar los **ejes de preferencia** (PCA) y **visualizar** la estructura (MDS / t-SNE).
* **Agrupar** con varios algoritmos y **combinarlos** con la reducción de dimensión, comparándolos.
* **Elegir el número de grupos** con criterios objetivos (codo, silueta, dendrograma, índices internos).
* **Caracterizar** cada segmento (qué categorías lo definen) y **nombrarlo** (amantes de la cultura, de la playa, del ocio nocturno…).
* Construir una **aplicación interactiva (widget) dentro de Colab** que explote la segmentación.

## <font color="steelblue">El conjunto de datos</font>

**5.456 usuarios** × **24 categorías** de valoración (UCI *Travel Review Ratings*; Renjith et al., 2018). Cada valor es la **valoración media** del usuario en esa categoría, en **[0, 5]**. **Sin valores perdidos** (versión del curso).

* **`User`** — identificador de texto. **Eliminar antes de agrupar.**
* **`Category 1`…`Category 24`** — iglesias, complejos turísticos, playas, parques, teatros, museos, centros comerciales, zoológicos, restaurantes, bares, servicios locales, hamburgueserías/pizzerías, hoteles, zumerías, galerías de arte, discotecas, piscinas, gimnasios, panaderías, belleza/spas, cafeterías, miradores, monumentos, jardines.

> **Semántica del 0 (importante):** un **0** indica **ausencia de valoraciones** en esa categoría, **no** "muy mala valoración". Decidid cómo tratarlo y cómo afecta a las distancias del *clustering*.
> **Renombrad las categorías** con sus nombres temáticos: la **interpretación** de componentes y segmentos será mucho más clara.

## <font color="steelblue">Reglas del juego</font>

1. **Quita `User`** y trabaja solo con las 24 valoraciones.
2. **Escala antes de agrupar.** El *clustering* y la PCA se basan en distancias/varianzas: estandariza (o justifica no hacerlo). Documenta el efecto.
3. **Decide la semántica del 0** (ausencia de valoración) y su impacto.
4. **Combina modelos:** no te quedes en un solo algoritmo; **reduce dimensión + agrupa**, compara **varios** métodos de *clustering* y contrasta sus soluciones.
5. **Elige el número de grupos con criterios** (codo, silueta, dendrograma, índices), no a ojo.
6. **El objetivo es interpretar:** un *clustering* sin **caracterización** de los grupos no vale. Nombra y describe cada segmento.
7. **Reproducibilidad:** fija `random_state` y comenta la **estabilidad** de la solución (el no supervisado es sensible a la inicialización).

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
url = "https://raw.githubusercontent.com/ia4legos/MachineLearning/main/data/google_review_ratings.csv"
reviews = pd.read_csv(url)
print(f"Dimensiones: {reviews.shape[0]:,} usuarios × {reviews.shape[1]} columnas")
reviews.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas a desarrollar**
1. **Estructura.** Confirmad 24 categorías + `User`; **renombradlas** con sus nombres temáticos (iglesias, playas, museos…).
2. **Semántica del 0.** Cuantificad cuántos 0 hay por categoría y discutid qué significan (ausencia de valoración).
3. **Distribuciones.** Histogramas/boxplots por categoría: ¿qué categorías se valoran más/menos? ¿hay asimetrías?
4. **Correlaciones.** Matriz de correlación entre categorías: ¿se aprecian **bloques temáticos** (cultura, ocio, gastronomía, naturaleza…)? Esto anticipa los ejes de la PCA.
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** ¿por qué hay que **escalar** antes de calcular distancias o componentes? ¿qué pasaría con categorías de mayor varianza si no lo hicieras?

# <font color="steelblue">Fase 2 — Preprocesado y escalado</font>

**Tareas a desarrollar**
1. **Quita `User`** y construye la matriz `X` de 24 valoraciones.
2. **Escalado:** estandariza (`StandardScaler`) o normaliza; **justifica** la elección y observa su efecto sobre las distancias.
3. **(Opcional) Tratamiento del 0** y de posibles atípicos.
4. Mantén una versión **escalada** para PCA/clustering y conserva los **valores originales** para la **interpretación** de los grupos (Fase 6).

> **A responder:** ¿estandarizar (media 0, varianza 1) o normalizar por usuario? ¿qué cambia en la interpretación?

# <font color="steelblue">Fase 3 — Reducción de dimensión</font>

**Tareas a desarrollar**
1. **PCA.** Ajusta PCA sobre `Xs`; dibuja el **scree plot** (varianza explicada y acumulada) y decide cuántas componentes retienes.
2. **Interpretación de componentes (clave).** Examina los **loadings** de las primeras componentes: ¿qué categorías cargan en PC1, PC2…? Dales un **significado** (p. ej. "eje cultura ↔ ocio").
3. **Visualización.** Proyecta los usuarios en PC1–PC2 y, además, con **MDS** y/o **t-SNE**/**UMAP**, para ver si hay estructura/agrupamientos naturales.
4. **Conclusión:** ¿la nube tiene estructura? ¿cuántas dimensiones "reales" parece haber?

> **Combinar modelos:** podrás **agrupar sobre el espacio PCA** (menos ruido) y comparar con agrupar sobre las 24 variables originales (Fase 4).

# <font color="steelblue">Fase 4 — Agrupación (clustering) y combinación de modelos</font>

**Tareas a desarrollar**
1. Aplica y compara **al menos tres** algoritmos del curso: **K-Means**, **aglomerativo/jerárquico**, y **DBSCAN** y/o **mixturas gaussianas (GMM)**.
2. **Combina con la reducción de dimensión:** agrupa sobre el **espacio PCA** y compáralo con agrupar sobre las variables escaladas. ¿Cambian los grupos?
3. Para el jerárquico, dibuja el **dendrograma**; para DBSCAN, explora `eps`/`min_samples`; para GMM, prueba distintas covarianzas.
4. **Compara las particiones** entre algoritmos (p. ej. concordancia con *Adjusted Rand Index* entre soluciones).

> **A responder:** ¿qué algoritmo da grupos más nítidos e interpretables aquí? ¿K-Means asume grupos esféricos: es razonable con estos datos?

# <font color="steelblue">Fase 5 — Validación y elección del número de grupos</font>

**Tareas a desarrollar**
1. **Número de grupos.** Usa el **método del codo** (inercia) y el **coeficiente de silueta** para K-Means; el **dendrograma** para el jerárquico.
2. **Índices internos.** Compara soluciones con **silueta**, **Davies–Bouldin** y **Calinski–Harabasz**.
3. **Estabilidad.** Repite con varias semillas/submuestras y comprueba si la solución es **estable** (el no supervisado es sensible a la inicialización).
4. **Decisión justificada** del algoritmo y el número de segmentos finales.

> **Aviso:** no existe "la verdad"; un buen número de grupos equilibra **calidad estadística** (índices) e **interpretabilidad** (Fase 6).

# <font color="steelblue">Fase 6 — Caracterización de los grupos (objetivo central)</font>

Agrupar no basta: hay que **entender** y **nombrar** los segmentos.

**Tareas a desarrollar**
1. **Perfiles.** Calcula la **valoración media por categoría** en cada grupo (sobre los **valores originales**) y construye un **mapa de calor** grupos × categorías.
2. **Rasgos distintivos.** Para cada grupo, identifica las categorías en las que **destaca** respecto a la media global (z-score por grupo) y, opcionalmente, un **radar/spider plot**.
3. **Tamaño y nombre.** Indica cuántos usuarios tiene cada segmento y **ponle un nombre interpretable** (p. ej. "amantes de la cultura", "playa y ocio", "gastronómicos urbanos"…).
4. **Lectura de negocio.** ¿Cómo usaría una agencia/portal turístico esta segmentación (recomendación, campañas)?

> Es la parte con **más peso**: la calidad de la **interpretación** define el proyecto.

# <font color="steelblue">Fase 7 — Aplicación interactiva (widget en Colab)</font>

El objetivo no es desplegar un servicio, sino una **aplicación directa dentro del cuaderno** con **`ipywidgets`** que explote la segmentación. Implementa **al menos una**:

1. **Explorador de segmentos:** un desplegable que, al elegir un segmento, muestre su **perfil** (radar/barras), su tamaño y sus categorías más características.
2. **Asignador + recomendador:** *sliders* para que un usuario nuevo introduzca sus valoraciones; el widget **escala → proyecta → asigna el segmento más cercano** y **recomienda** las categorías mejor valoradas por ese segmento que el usuario aún no frecuenta.
3. **(Opcional)** Mapa interactivo PC1–PC2 coloreado por segmento (con `plotly`).

> Mantén el preprocesado/modelo ya ajustados en memoria; el widget solo los **usa** (no reentrena en cada interacción).

# <font color="steelblue">Entregables y formato</font>

* **Cuaderno** ejecutable de principio a fin, con código y **celdas de texto** que justifiquen cada decisión (escalado, nº de grupos, interpretación).
* **Memoria breve:** EDA, reducción de dimensión e interpretación de ejes, comparación de algoritmos de *clustering*, validación y elección, **caracterización y nombres de los segmentos**, y la aplicación.
* **Widget(s)** funcionando dentro del cuaderno.
* **Reproducibilidad:** `random_state` fijado y comentario de estabilidad.

## <font color="steelblue">Rúbrica de evaluación (100 pts)</font>

| Fase | Qué se valora | Puntos |
|---|---|---:|
| 1. EDA | Renombrado, semántica del 0, distribuciones, bloques de correlación | 12 |
| 2. Preprocesado y escalado | Escalado justificado, original vs escalado para interpretar | 12 |
| 3. Reducción de dimensión | PCA (varianza + **interpretación de loadings**), MDS/t-SNE | 16 |
| 4. Clustering y combinación | ≥3 algoritmos, **clustering sobre PCA vs original**, comparación | 16 |
| 5. Validación y nº de grupos | Codo/silueta/dendrograma, índices internos, **estabilidad** | 14 |
| 6. Caracterización de grupos | Perfiles, rasgos distintivos, **nombres**, lectura de negocio | 18 |
| 7. Aplicación (widget) | `ipywidgets` funcional (explorador y/o recomendador) | 12 |
| — Extra | UMAP, **consenso de clustering**, *gap statistic*, validación de estabilidad formal, mapa `plotly` | +10 |

> **Penalizaciones:** **no escalar** antes de agrupar/PCA, elegir el nº de grupos **a ojo** sin criterios, quedarse en las etiquetas **sin caracterizar** los segmentos, o tratar el **0 como mala valoración** sin discutirlo.

# <font color="steelblue">Pistas y errores típicos</font>

* **Escala primero.** Sin escalar, las categorías de mayor varianza dominan las distancias y la PCA.
* **Interpreta la PCA.** Los *loadings* convierten PC1/PC2 en **ejes con significado** (cultura↔ocio, urbano↔naturaleza): es la base para nombrar segmentos.
* **No te cases con K-Means.** Asume grupos esféricos y de tamaño similar; compara con jerárquico/GMM/DBSCAN.
* **Elige k con criterios + sentido.** El mejor k estadístico no siempre es el más interpretable; busca el equilibrio.
* **Caracterizar es el objetivo.** Un número de grupo no dice nada; el **perfil** y el **nombre** sí.
* **Widget, no despliegue.** Usa `ipywidgets` dentro de Colab; no hace falta `joblib`/servidor.

# <font color="steelblue">Referencias</font>

* Renjith, S., Sreekumar, A. & Jathavedan, M. (2018). *Evaluation of Partitioning Clustering Algorithms for Processing Social Media Data in Tourism Domain*. IEEE RAICS.
* Renjith, S. (2018). *Travel Review Ratings*. UCI ML Repository (id 485).
* Cuadernos del curso: *Componentes principales y variantes*, *Escalas multidimensionales (MDS)*, *Métodos de manifold*, *Clustering (K-Means, jerárquico, DBSCAN, mixturas gaussianas)*.
